# 00 — Admin2 hazard valid-pixel counts (weight table for aggregation)

Standalone, lightweight GEE helper that produces a per-(admin2 × hazard) **valid-pixel count** (`hazs_count`). This count is the correct weight for aggregating `hazs_mean` / `hazs_std` up to admin1 / admin0 (see notebooks 06 and 07): the true spatial mean is pixel-weighted, and pooled variance is exactly reconstructable from `(count, mean, std)` per child.

**Why separate from notebook 01:** this runs *only* an `ee.Reducer.count()` over the raw hazard image — no population stack, no exposure, no multi-statistic reducer — so it is far cheaper than a full re-extraction. It does **not** modify `GEE_functions.py` or the main `combined_reducer`. Run it opportunistically; the aggregation pipelines fall back to a population weight for any hazard/unit missing here.

## Output

`{OUTPUT_FOLDER}/aux/hazs_count_{date}.csv` — tidy table: `adm2_ucode, hazard_code, hazs_count`.

## Notes

- Counts pixels of the **raw** hazard image within each admin2 polygon, after the same `apply_nodata` / theoretical-minimum masking that `define_hazard` performs (so it matches the domain over which `hazs_mean`/`std` were computed in notebook 01).
- **Scale = the country's population/analysis scale** (WorldPop, ~92.77 m) — the same `analysis_scale` notebook 01 passes to its `hazs_*` zonal reducers. The weight must count the *same pixel set* the mean was computed over, so the two must share a scale. `SCALE_OVERRIDE` forces a fixed scale instead.

## ⚠️ Do not use the hazard image's own `nominalScale()`

Three hazards are mosaics (`mosaic: 1` in `hazard_info.json`):
`river_flood_100yr`, `coastal_flood_100yr`, `tropical_storm_100yr`.

`ee.ImageCollection.mosaic()` returns an image with **no fixed projection**, so
`nominalScale()` reports the WGS84 default of **111319.49 m (~1°)** rather than the
raster's true resolution (92.8 m, 92.7 m and 11132 m respectively). Reducing at
~111 km samples only a handful of points per admin2, collapsing `hazs_count` to
0–3. A zero count becomes a zero aggregation weight, which blanks `hazs_mean`
at admin1 for every affected governorate.

This was the cause of the 44 blank admin1 means in the Egypt `2026-08-31` run
(19 river flood, 12 tropical storm, 11 coastal flood, 2 ACLED). Verified against
GEE: Beheira showed 0 river-flood pixels at 111 km but **533,040** at 92.8 m.

The adm2 `hazs_mean` values themselves were never affected — notebook 01 reduces
at `analysis_scale`, not at the mosaic scale — so only this weight table had to
be rebuilt.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
LIB_DIR = PROJECT_ROOT / "lib"
CONFIG_DIR = PROJECT_ROOT / "config"
if str(LIB_DIR) not in sys.path:
    sys.path.insert(0, str(LIB_DIR))

import os
import datetime as dt
import pandas as pd
import ee

from GEE_functions import GEEUtils, Hazard

pd.set_option("display.max_columns", 500)

In [ ]:
# Credentials are cached after a one-time `earthengine authenticate` in a terminal,
# so Initialize alone works. Only re-authenticate if Initialize raises.
try:
    ee.Initialize(project="unicef-ccri")
except Exception:
    ee.Authenticate()
    ee.Initialize(project="unicef-ccri")

unicef_data_source_path = "projects/unicef-ccri/assets"
utils = GEEUtils(ee)

In [ ]:
# ============================================================
# Configuration
# ============================================================
OUTPUT_FOLDER = "output"          # where aux/ table is written
PROD_DATE = dt.date.today().isoformat()

# Optional filters (None = all)
PROCESS_SPECIFIC_COUNTRIES = ["EGY"]   # e.g. ["EGY"]
PROCESS_SPECIFIC_HAZARDS = None     # e.g. ["river_flood_100yr"]

# Reduction scale (metres).
#
# None  -> use the country's population/analysis scale (WorldPop, ~92.77 m),
#          which is the scale notebook 01 reduces hazs_mean/std at. This is the
#          correct default: the weight must count the SAME pixel set the mean
#          was computed over.
#
# Do NOT fall back to the hazard image's own nominalScale(): for the mosaicked
# hazards (river_flood_100yr, coastal_flood_100yr, tropical_storm_100yr)
# ImageCollection.mosaic() has no fixed projection, so nominalScale() returns
# the WGS84 default of 111319.49 m (~1 degree). Counting at that scale samples
# a handful of points per admin2 and collapses hazs_count to 0-3, which then
# zeroes the aggregation weight and blanks hazs_mean at admin1.
SCALE_OVERRIDE = None

TILE_SCALE = 4

os.makedirs(f"{OUTPUT_FOLDER}/aux", exist_ok=True)

In [ ]:
hazard_info = pd.read_json(CONFIG_DIR / "hazard_info.json")
hazard_list = [
    Hazard.from_dict(row, asset_prefix=unicef_data_source_path)
    for row in hazard_info.to_dict(orient="records")
]
countries_to_process = utils.get_countries_to_process(PROCESS_SPECIFIC_COUNTRIES)
print(f"Hazards: {len(hazard_list)} | Countries: {len(countries_to_process)}")

In [ ]:
# ============================================================
# Build a non-blocking Export.table.toDrive task for one hazard x country.
# No getInfo() -> tasks run server-side in parallel instead of serially.
# ============================================================

# Hazards whose raw-image pixel count is identical to another hazard's, because
# they share the same asset/nodata/th_min and differ ONLY by threshold (which
# count() does not use). key = duplicate code, value = source code to copy from.
DUPLICATE_COUNT_OF = {
    "earthquake_pga_0p34": "earthquake_pga_0p09",
    "landslides_0p01":     "landslides_0p1",
}

DRIVE_FOLDER = "hazs_count_exports_fixed"   # single Drive folder for all task outputs

_ANALYSIS_SCALE_CACHE = {}   # country_ucode -> ee.Number | None


In [ ]:


def get_analysis_scale(country_ucode):
    """Return the country's population/analysis scale (WorldPop, ~92.77 m).

    This is the scale notebook 01 uses for the hazs_mean/std zonal reducers
    (GEEUtils.analysis_scale, set from the population raster in
    get_population_classes). Counting at the same scale is what makes
    hazs_count a valid weight for those means.

    Cached per country: the scale depends only on the country's population
    raster, not on the hazard, so it is resolved once per country instead of
    once per (country x hazard).

    Returns None for countries with no population raster; the caller then
    falls back to the hazard's true native scale.
    """
    if country_ucode in _ANALYSIS_SCALE_CACHE:
        return _ANALYSIS_SCALE_CACHE[country_ucode]

    if utils.check_country_no_population(country_ucode):
        scale = None
    else:
        adm2_fc = utils.get_admin2_boundaries(country_ucode, admin_level="adm0_ucode")
        utils.set_population_dicts(country_ucode, adm2_fc)
        utils.get_population_classes()      # sets utils.analysis_scale
        scale = utils.analysis_scale

    _ANALYSIS_SCALE_CACHE[country_ucode] = scale
    return scale


def get_native_scale(hazard):
    """Native scale of the hazard raster, safe for mosaicked assets.

    ImageCollection.mosaic() drops the projection, so nominalScale() on the
    mosaic returns the WGS84 default (111319.49 m). Take the scale from a
    member image instead. Only used as a fallback for no-population countries.
    """
    if hazard.mosaic:
        return ee.Image(ee.ImageCollection(hazard.asset).first()) \
                 .projection().nominalScale()
    return ee.Image(hazard.asset).projection().nominalScale()


def submit_hazard_pixel_count_export(
    hazard,
    country_ucode,
    scale_override=None,
    tile_scale=8,
    drive_folder=DRIVE_FOLDER,
):
    """Submit ONE non-blocking Export.table.toDrive task counting valid raw-hazard
    pixels per admin2 unit for `hazard` x `country_ucode`.

    Uses only ee.Reducer.count() over the raw (nodata-masked) hazard image — the
    same image domain define_hazard produces — so counts match the pixels that
    fed hazs_mean/std in notebook 01.

    Scale: the country's population/analysis scale, matching notebook 01's
    reducers. NOT the hazard image's nominalScale(), which is wrong (111 km)
    for the three mosaicked hazards. See the config cell.

    Returns the started ee.batch.Task, or None if skipped as a duplicate.
    Output CSV is named {country}__{hazard_code}.csv so results reconstruct
    per-country.
    """
    # Skip duplicates: their count is identical to the source hazard's.
    if hazard.code in DUPLICATE_COUNT_OF:
        return None

    images = utils.define_hazard(hazard)
    raw = images["hazard_raw_image"].rename("haz").toFloat()

    adm2_fc = utils.get_admin2_boundaries(country_ucode, admin_level="adm0_ucode")

    if scale_override:
        scale = scale_override
    else:
        # Match notebook 01's reduction scale; fall back to the hazard's true
        # native scale only where there is no population raster.
        scale = get_analysis_scale(country_ucode) or get_native_scale(hazard)

    reduced = raw.reduceRegions(
        collection=adm2_fc,
        reducer=ee.Reducer.count().setOutputs(["hazs_count"]),
        scale=scale,
        tileScale=tile_scale,
    ).select(["adm2_ucode", "hazs_count"], None, False)

    # Tag rows with the hazard code server-side so the CSV is self-describing.
    reduced = reduced.map(lambda f: f.set("hazard_code", hazard.code))

    desc = f"cnt_{country_ucode}_{hazard.code}"[:100]  # EE description limit
    fname = f"{country_ucode}__{hazard.code}"
    task = ee.batch.Export.table.toDrive(
        collection=reduced,
        description=desc,
        folder=drive_folder,
        fileNamePrefix=fname,
        fileFormat="CSV",
        selectors=["adm2_ucode", "hazard_code", "hazs_count"],
    )
    task.start()
    return task

In [ ]:
# ============================================================
# Submit export tasks, looping COUNTRY -> HAZARD so you can run pieces
# (e.g. EGY first) and reconstruct the table per-country as tasks finish.
#
# EE caps pending tasks (~3000). 21 hazards x ~200 countries would exceed that,
# so submit in country batches: set SUBMIT_COUNTRIES to your priority list,
# run, collect the CSVs, then move to the next batch.
# ============================================================

# IN ORDER. A list runs just those countries; None = every country.
SUBMIT_COUNTRIES = None# ["EGY_V1"]   # <-- set to `countries_to_process` for the full run

_targets = SUBMIT_COUNTRIES if SUBMIT_COUNTRIES else countries_to_process

submitted = []   # (country, hazard_code, task_id)
for country_ucode in _targets:
    print(f"\n=== Submitting {country_ucode} ===")
    for hazard in hazard_list:
        if PROCESS_SPECIFIC_HAZARDS and hazard.code not in PROCESS_SPECIFIC_HAZARDS:
            continue
        try:
            task = submit_hazard_pixel_count_export(hazard, country_ucode, SCALE_OVERRIDE)
            if task is None:
                print(f"  {hazard.code}: skipped (duplicate of "
                      f"{DUPLICATE_COUNT_OF[hazard.code]})")
                continue
            submitted.append((country_ucode, hazard.code, task.id))
            print(f"  {hazard.code}: task {task.id} started")
        except Exception as e:
            print(f"  ERROR {hazard.code} / {country_ucode}: {e}")

print(f"\nSubmitted {len(submitted)} export tasks to Drive folder '{DRIVE_FOLDER}'.")
print("Monitor at https://code.earthengine.google.com/tasks or run the next cell.")

# Manifest for tracking / reconstruction.
manifest_path = f"{OUTPUT_FOLDER}/aux/export_manifest_{PROD_DATE}.csv"
pd.DataFrame(submitted, columns=["country", "hazard_code", "task_id"]).to_csv(
    manifest_path, index=False)
print(f"Manifest -> {manifest_path}")

In [ ]:
# ============================================================
# (Optional) Poll the submitted tasks until they finish.
# Safe to skip — you can also watch https://code.earthengine.google.com/tasks
# ============================================================
import time

def poll_tasks(task_ids, every=30, timeout=3600):
    pending = set(task_ids)
    start = time.time()
    while pending and (time.time() - start) < timeout:
        for t in ee.batch.Task.list():
            if t.id in pending and t.state in ("COMPLETED", "FAILED", "CANCELLED"):
                pending.discard(t.id)
                print(f"{t.state:10s} {t.id}")
        if pending:
            print(f"  {len(pending)} still running... ({int(time.time()-start)}s)")
            time.sleep(every)
    print("Done." if not pending else f"Timeout with {len(pending)} still pending.")

poll_tasks([t for _, _, t in submitted])

In [ ]:
# ============================================================
# ASSEMBLE the tidy count table from the downloaded Drive CSVs.
# Run AFTER the tasks complete and you've downloaded the Drive folder
# 'hazs_count_exports' into DRIVE_CSV_DIR below.
#
# Reconstructs the 2 duplicate hazards (earthquake_pga_0p34, landslides_0p01)
# from their twins so the final table has all 23 hazard codes.
# ============================================================
import glob

DRIVE_CSV_DIR = f"{OUTPUT_FOLDER}/{DRIVE_FOLDER}"   # where you put the downloaded CSVs

files = glob.glob(f"{DRIVE_CSV_DIR}/*.csv")
if not files:
    print(f"No CSVs in {DRIVE_CSV_DIR}. Download the '{DRIVE_FOLDER}' Drive folder there first.")
else:
    counts_table = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

    # Add duplicate hazards back in, copied from their source counts.
    for dup, src in DUPLICATE_COUNT_OF.items():
        twin = counts_table[counts_table.hazard_code == src].copy()
        if twin.empty:
            print(f"WARNING: source '{src}' not found; cannot build duplicate '{dup}'.")
            continue
        twin["hazard_code"] = dup
        counts_table = pd.concat([counts_table, twin], ignore_index=True)

    counts_table = counts_table[["adm2_ucode", "hazard_code", "hazs_count"]]
    out_path = f"{OUTPUT_FOLDER}/aux/hazs_count_{PROD_DATE}.csv"
    counts_table.to_csv(out_path, index=False)
    print(f"Wrote {len(counts_table)} rows ({counts_table.hazard_code.nunique()} hazards) to {out_path}")
    counts_table.head()